# Exercise: Classical Time-Series Models

You work for a regional electric utility. Every month, you forecast electricity demand so grid operators can schedule generation capacity. Over-forecast and you waste fuel. Under-forecast and you risk brownouts.

You have eight years of monthly demand data plus average temperatures. Demand is U-shaped with temperature — high in summer (AC) and winter (heating), low in spring and fall.

Your task: fit ARIMA, SARIMAX, Prophet, and Darts ARIMA models, run diagnostics, and compare forecasts with prediction intervals.

## Data

`../data/electricity_demand.csv` — monthly demand (MWh) and temperature (F) from January 2017 through December 2024

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from prophet import Prophet
from darts import TimeSeries
from darts.models import ARIMA as DartsARIMA
from scipy import stats

HORIZON = 12

## Load and Explore Data

In [ ]:
df = pd.read_csv("../data/electricity_demand.csv", parse_dates=["date"], index_col="date")
df = df.asfreq("MS")
demand = df["demand_mwh"]
temp = df["avg_temp_f"]

print(f"Data: {len(df)} months, {df.index[0].date()} to {df.index[-1].date()}")
print(f"Demand range: {demand.min():,.0f} to {demand.max():,.0f} MWh")
print(f"Temperature range: {temp.min():.1f} to {temp.max():.1f} F")

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
axes[0].plot(demand.index, demand / 1e6, color="black")
axes[0].set_ylabel("Demand (M MWh)")
axes[0].set_title("Monthly Electricity Demand")
axes[1].plot(temp.index, temp, color="tab:orange")
axes[1].set_ylabel("Avg Temp (F)")
plt.tight_layout()
plt.show()

## Train/Test Split and ACF/PACF

In [ ]:
# Train/test split and ACF/PACF
train_end = "2023-12-01"
train_df = df.loc[:train_end]
test_df = df.loc[train_end:].iloc[1:]
train_demand = train_df["demand_mwh"]
train_temp = train_df["avg_temp_f"]

# Seasonal average temperature for future exogenous
seasonal_avg_temp = train_df.groupby(train_df.index.month)["avg_temp_f"].mean()
future_temp = pd.Series(
    [seasonal_avg_temp[m] for m in range(1, 13)],
    index=pd.date_range("2024-01-01", periods=12, freq="MS"),
)

# ACF/PACF of seasonally differenced series
diffed = train_demand.diff(12).dropna()
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(diffed, ax=axes[0], lags=36)
plot_pacf(diffed, ax=axes[1], lags=36)
axes[0].set_title("ACF (seasonally differenced)")
axes[1].set_title("PACF (seasonally differenced)")
plt.tight_layout()
plt.show()

## Your Task

Implement the six modeling functions below:

1. **`fit_arima`** — fit ARIMA(2,1,1) using statsmodels SARIMAX
2. **`fit_sarimax`** — fit SARIMAX(1,1,1)(1,1,1,12) with temperature exogenous
3. **`forecast_with_intervals`** — generate forecast mean + confidence intervals
4. **`run_residual_diagnostics`** — compute AIC, Ljung-Box p-value, normality p-value
5. **`fit_prophet`** — fit Prophet with temperature as regressor
6. **`fit_darts_arima`** — fit ARIMA via the Darts framework

In [ ]:
def fit_arima(train, order=(2, 1, 1)):
    """Fit an ARIMA model using statsmodels SARIMAX."""
    # TODO: implement
    raise NotImplementedError


def fit_sarimax(train, exog, order=(1, 1, 1), seasonal_order=(1, 1, 1, 12)):
    """Fit a SARIMAX model with exogenous variable."""
    # TODO: implement
    raise NotImplementedError


def forecast_with_intervals(fitted, steps, exog_future=None, alpha=0.05):
    """Return (predicted_mean, conf_int_dataframe) from a fitted model."""
    # TODO: implement
    raise NotImplementedError


def run_residual_diagnostics(fitted):
    """Return dict with ljung_box_p, normality_p, aic, bic."""
    # TODO: implement
    raise NotImplementedError


def fit_prophet(df):
    """Fit a Prophet model with avg_temp_f as regressor. Return the fitted model."""
    # TODO: implement
    raise NotImplementedError


def fit_darts_arima(series, order=(2, 1, 1)):
    """Fit an ARIMA model via Darts. Return (model, TimeSeries)."""
    # TODO: implement
    raise NotImplementedError

## Fit and Compare Models

In [ ]:
# Fit and evaluate all models
arima_fit = fit_arima(train_demand)
arima_diag = run_residual_diagnostics(arima_fit)
arima_fc, arima_ci = forecast_with_intervals(arima_fit, HORIZON)

sarimax_fit = fit_sarimax(train_demand, train_temp)
sarimax_diag = run_residual_diagnostics(sarimax_fit)
sarimax_fc, sarimax_ci = forecast_with_intervals(
    sarimax_fit, HORIZON, exog_future=future_temp.values
)

prophet_model = fit_prophet(train_df)
future_dates = prophet_model.make_future_dataframe(periods=HORIZON, freq="MS")
future_dates["avg_temp_f"] = list(train_df["avg_temp_f"].values) + list(future_temp.values)
prophet_fc = prophet_model.predict(future_dates)
prophet_pred = prophet_fc.iloc[-HORIZON:]

darts_model, darts_ts = fit_darts_arima(train_demand)
darts_fc = darts_model.predict(HORIZON)

print("=== Model Comparison ===")
print(f"{'Model':<35} {'AIC':>10} {'LB p':>10} {'Norm p':>10}")
print("-" * 67)
print(f"{'ARIMA(2,1,1)':<35} {arima_diag['aic']:>10.1f} {arima_diag['ljung_box_p']:>10.4f} {arima_diag['normality_p']:>10.4f}")
print(f"{'SARIMAX(1,1,1)(1,1,1,12) + temp':<35} {sarimax_diag['aic']:>10.1f} {sarimax_diag['ljung_box_p']:>10.4f} {sarimax_diag['normality_p']:>10.4f}")

## Forecast Visualization

In [ ]:
# Plot forecast comparison
forecasts_dict = {
    "ARIMA(2,1,1)": (arima_fc, arima_ci.iloc[:, 0], arima_ci.iloc[:, 1]),
    "SARIMAX + temperature": (sarimax_fc, sarimax_ci.iloc[:, 0], sarimax_ci.iloc[:, 1]),
    "Prophet + temperature": (
        pd.Series(prophet_pred["yhat"].values, index=sarimax_fc.index),
        prophet_pred["yhat_lower"].values,
        prophet_pred["yhat_upper"].values,
    ),
}

n = len(forecasts_dict)
fig, axes = plt.subplots(n, 1, figsize=(14, 5 * n), sharex=True)
colors = ["tab:blue", "tab:orange", "tab:green"]
for ax, (name, (mean, lower, upper)), color in zip(axes, forecasts_dict.items(), colors):
    ax.plot(train_demand.index, train_demand.values, color="black", label="Historical")
    ax.plot(mean.index, mean.values, color=color, linestyle="--", linewidth=2, label="Forecast")
    if lower is not None and upper is not None:
        ax.fill_between(mean.index, lower, upper, color=color, alpha=0.15, label="95% interval")
    ax.set_ylabel("Demand (MWh)")
    ax.set_title(name)
    ax.legend(loc="upper left")
plt.tight_layout()
plt.show()